# فصل چهارم: پیش‌پردازش، نمونه‌گیری و کاهش بُعد

این Notebook از داده‌های **کاملاً ساختگی** و الگوریتم‌های پیوست فصل استفاده می‌کند. متن فارسی توضیح است و نمادهای داخل فرمول‌ها انگلیسی هستند. همهٔ خروجی‌ها آزمایشی‌اند.

In [1]:
import numpy as np
import pandas as pd
from chapter04_examples import *
print("Notebook and core functions loaded")

Notebook and core functions loaded


## ۱. بازنمایی تراکنش و تجمیع روزانه

مجموع مبلغ قبل و بعد از تجمیع باید ثابت بماند، ولی سطح مشاهده تغییر می‌کند.

In [2]:
df = generate_transactions()
daily = aggregate_day(df)
assert np.isclose(df.spend.sum(), daily.total_spend.sum())
print("rows:", len(df), "days:", len(daily), "total:", round(df.spend.sum(), 2))
print(daily.head(3).to_string(index=False))

rows: 320 days: 45 total: 42011.06
 day  total_spend  n_transactions  mean_spend  rare_events
   0      1183.16              11  107.560000            0
   1       540.93               5  108.186000            0
   2      1416.21               9  157.356667            2


## ۲. نمونه‌گیری و رخداد نادر

نمونهٔ تصادفی الزاماً همهٔ الگوهای نادر را پوشش نمی‌دهد. این بذر ثابت در این آزمایش تنها یک نمونهٔ مشخص تولید می‌کند.

In [3]:
sample = srs_without_replacement(df, 50)
print("rare events in full data:", int(df.rare_event.sum()))
print("rare events in 50-row sample:", int(sample.rare_event.sum()))
assert len(set(sample.transaction_id)) == 50

rare events in full data: 12
rare events in 50-row sample: 0


## ۳. نمونه‌گیری طبقه‌ای و وزن تصحیح

اگر در جامعه سهم گروه‌ها برابر نباشد، میانگین بی‌وزنِ نمونهٔ نامتناسب، میانگین جامعه را برآورد نمی‌کند.

In [4]:
toy = pd.DataFrame({"group": ["A"]*8+["B"]*2, "x": [1]*8+[9]*2})
s = stratified_sample(toy, "group", {"A":2, "B":2})
print("population mean:", toy.x.mean(), "unweighted sample mean:", s.x.mean())
print("correctly weighted mean:", weighted_stratum_mean(s, toy, "group", "x"))
assert weighted_stratum_mean(s,toy,"group","x") == toy.x.mean()

population mean: 2.6 unweighted sample mean: 5.0
correctly weighted mean: 2.6


## ۴. مقیاس‌بندی و جلوگیری از نشت اطلاعات

آماره‌های تبدیل روی بخش آموزش fit و روی آزمون transform می‌شوند.

In [5]:
x = df[["spend", "units", "visits30", "discount_pct"]].to_numpy()
train, test = x[:240], x[240:]
p = zscore_fit(train)
train_z, test_z = zscore_transform(train,p), zscore_transform(test,p)
assert np.allclose(train_z.mean(axis=0), 0, atol=1e-10)
print("train mean (z):", np.round(train_z.mean(axis=0), 5))
print("test mean (z):", np.round(test_z.mean(axis=0), 5))

train mean (z): [ 0.  0.  0. -0.]
test mean (z): [ 0.0741  -0.11345 -0.11279  0.09637]


## ۵. PCA دو مؤلفه‌ای با برازش روی آموزش

سهم واریانس و خطای بازسازی برای این دادهٔ ساختگی گزارش می‌شود؛ این شاخص‌ها معادل حفظ هر الگوی نادر نیستند.

In [6]:
pca = pca_fit(train_z,2)
z = pca_transform(test_z,pca)
ratio = pca_variance_ratio(pca)
print("explained variance ratio:", np.round(ratio,4))
print("top-2 cumulative:", round(float(ratio[:2].sum()),6))
print("test reconstruction MSE:", round(reconstruction_mse(test_z,pca),6))
print("test shape:", z.shape)
assert z.shape==(80,2)

explained variance ratio: [0.3854 0.2858 0.2193 0.1095]
top-2 cumulative: 0.671211
test reconstruction MSE: 0.351702
test shape: (80, 2)


## ۶. حفظ همسایگی بعد از فرافکنی

همسایگان قبل و بعد از کاهش بُعد یکسان نیستند. شاخص زیر میانگین جاکارد مجموعه‌های همسایگی است.

In [7]:
test_overlap = neighbor_overlap(test_z, z, neighbors=5)
print("neighbor Jaccard overlap (5-NN):", round(test_overlap,4))
assert 0 <= test_overlap <= 1

neighbor Jaccard overlap (5-NN): 0.3765


## ۷. آزمایش دستی مقدارهای ویژه و SVD

در دادهٔ روی یک خط، PCA یک‌مؤلفه‌ای بازسازی دقیق انجام می‌دهد. تقریب ماتریس قطری نمونهٔ دوم است.

In [8]:
manual = np.array([[-2.,-1.],[-1,-.5],[0,0],[1,.5],[2,1]])
manual_pca=pca_fit(manual,1)
print("covariance:\n", (manual.T@manual)/len(manual))
print("eigenvalues:", manual_pca.eigenvalues)
print("manual reconstruction MSE:", reconstruction_mse(manual,manual_pca))
mat=np.diag([2.,1.]);print("SVD rank-1 error²:", round(float(np.linalg.norm(mat-best_rank_k_svd(mat,1),"fro")**2),6))

covariance:
 [[2.  1. ]
 [1.  0.5]]
eigenvalues: [2.5 0. ]
manual reconstruction MSE: 0.0
SVD rank-1 error²: 1.0


## ۸. جمع‌بندی محاسبات

این مجموعهٔ ساختگی ثابت، ابزار تدریس است نه یک دیتاست واقعی. تمام نتایج با داده، بذر و کد همراه فصل قابل تکرارند.

In [9]:
assert len(df)==320 and len(daily)==45
print(build_demo())

{'n_rows': 320, 'n_days': 45, 'sample_events': 0, 'population_events': 12, 'explained_variance': 0.6712105293321575, 'test_reconstruction_mse': 0.3517023963165604, 'test_projection_shape': (80, 2)}
